# Fine-tuning with BioImage.IO Fine-tune (`model-finetune`)

This tutorial drives the `model-finetune` BioEngine app end-to-end for **both** backends it serves:

- **micro-sam** (`vit_*` types) — Segment-Anything encoder + AIS instance-segmentation decoder.
- **Cellpose-SAM** (`cpsam`) and **Cellpose-DINO** (`cpdino`, `cpdino-vitb`).

You will connect to the service, inspect training capabilities, fine-tune on annotated image/label pairs, serve the fine-tuned checkpoint for inference, and export a draft BioImage.IO package.


## 1. Connect

The app exposes a Ray-Serve service. Connect with `hypha-rpc` and grab the service by id (`bioimage-io/model-finetune` on the public worker, or your own worker's id).

In [ ]:
from hypha_rpc import connect_to_server

server = await connect_to_server({"server_url": "https://hypha.aicell.io"})
svc = await server.get_service("bioimage-io/model-finetune")
print([m for m in dir(svc) if not m.startswith("_")])

## 2. Inspect training capabilities

`get_training_capabilities()` reports the available model families/sizes and the per-model GPU-memory requirement, so you can pick a model that fits the worker's GPU.

In [ ]:
caps = await svc.get_training_capabilities()
caps

## 3. Prepare data

Training needs **dense** instance-label masks paired 1:1 with the images (annotate every object per image). Images and labels may be passed as numpy arrays, http(s) URLs, or `get_upload_url` paths; labels may also be a `.geojson` polygon FeatureCollection. Here we use small numpy arrays as a placeholder — swap in your own data.

In [ ]:
import numpy as np

# Replace with your own images + dense instance masks (one integer id per object, 0 = background).
train_images = [np.random.randint(0, 255, (256, 256), dtype=np.uint8) for _ in range(3)]
train_labels = [np.zeros((256, 256), dtype=np.uint16) for _ in range(3)]  # <-- real annotations here
len(train_images), train_images[0].shape

## 4. Fine-tune Cellpose-SAM (`cpsam`)

`start_training` returns immediately with a `session_id`; training runs in a subprocess on a GPU replica. Poll `get_training_status` until the checkpoint is servable. `diam_mean` is the cellpose mean object diameter; the app also records the measured min/max/median training-cell diameter for you.

In [ ]:
res = await svc.start_training(
    train_images=train_images,
    train_labels=train_labels,
    model_type="cpsam",
    n_epochs=5,
    diam_mean=30.0,
    learning_rate=1e-5,
    label="tutorial-cpsam",
)
session_id = res["session_id"]
session_id

In [ ]:
import asyncio

while True:
    st = await svc.get_training_status(session_id=session_id)
    print(st["status"], st.get("message", ""))
    if st["status"] in ("COMPLETED", "FAILED", "STOPPED"):
        break
    await asyncio.sleep(10)
st

### Serve the fine-tuned model

Once `COMPLETED`, run inference from the session's checkpoint. For the cellpose backends you can tune mask generation through a single `postprocessing` dict (lower `cellprob_threshold` to pick up faint cells, raise `flow_threshold` to admit noisier masks).

In [ ]:
masks = await svc.infer(
    input_arrays=[train_images[0]],
    session_id=session_id,
    postprocessing={"cellprob_threshold": -1.0, "flow_threshold": 0.4},
)
masks[0]["output"]  # int32 [H, W] instance label mask (wire-dict)

### Export a draft BioImage.IO package

`export_model` builds a draft package (weights + wrapper, self-tested) and returns an export id; poll `get_export_status`. The export is a draft — publish it yourself when you are happy with it.

In [ ]:
exp = await svc.export_model(session_id=session_id, name="My fine-tuned Cellpose-SAM")
export_id = exp.get("export_id")

while True:
    es = await svc.get_export_status(export_id=export_id)
    print(es["status"])
    if es["status"] in ("COMPLETED", "FAILED"):
        break
    await asyncio.sleep(10)
es

## 5. Fine-tune micro-sam (`vit_t_lm`)

The micro-sam path is the same call with a `vit_*` `model_type`. It fine-tunes the AIS decoder; `n_objects_per_batch` is the main GPU-memory knob. After training you can also precompute image embeddings with `compute_embedding` and run AIS on them without re-encoding.

In [ ]:
res2 = await svc.start_training(
    train_images=train_images,
    train_labels=train_labels,
    model_type="vit_t_lm",
    n_epochs=5,
    n_objects_per_batch=8,
    patch_size=512,
    label="tutorial-microsam",
)
session2 = res2["session_id"]

while True:
    st2 = await svc.get_training_status(session_id=session2)
    print(st2["status"])
    if st2["status"] in ("COMPLETED", "FAILED", "STOPPED"):
        break
    await asyncio.sleep(10)
st2

In [ ]:
masks2 = await svc.infer(input_arrays=[train_images[0]], session_id=session2)
masks2[0]["output"]

## 6. Housekeeping

`list_training_sessions()` shows every session on the worker; `stop_training(session_id=...)` requests cancellation of a run in progress.

In [ ]:
sessions = await svc.list_training_sessions()
list(sessions)[:10]